In [0]:
# List files in your S3 bucket root
files = dbutils.fs.ls("s3://matchpulse-pawan/")
for file in files:
    print(f"{file.name} - {file.size} bytes")

In [0]:
%sql
-- See all external locations you have access to
SHOW EXTERNAL LOCATIONS;

In [0]:
# Get details about your external location
spark.sql("DESCRIBE EXTERNAL LOCATION `db_s3_external_databricks-s3-matchpulse`").display()

In [0]:
# List contents of the raw folder
raw_path = "s3://matchpulse-pawan/raw/"
print(f"Contents of {raw_path}:\n")

raw_files = dbutils.fs.ls(raw_path)
for item in raw_files:
    print(f"  {item.name} - {item.size} bytes")
    
    # If it's a folder (ends with /), list its contents too
    if item.name.endswith('/'):
        subfolder_path = item.path
        try:
            subfolder_files = dbutils.fs.ls(subfolder_path)
            for subitem in subfolder_files:
                print(f"    └─ {subitem.name} - {subitem.size} bytes")
        except Exception as e:
            print(f"    └─ Error listing: {str(e)}")

In [0]:
# Complete data summary and sample
import json

print("="*70)
print("COMPLETE STATSBOMB DATA STRUCTURE")
print("="*70)
print("\nraw/statsbomb/")
print("  ├── events/          417 files (~3 MB each)   - Match event data")
print("  ├── lineups/         417 files (~0.02 MB each) - Team lineups")
print("  └── matches/         3 subfolders             - Match metadata")
print("       ├── 11/90.json")
print("       ├── 16/90.json")
print("       └── 2/90.json")
print("\n" + "="*70)

# Sample a small lineup file to see the structure
print("\nSAMPLE DATA (lineups/22912.json):")
print("="*70)

lineup_sample = spark.read.option("multiLine", "true").json("s3://matchpulse-pawan/raw/statsbomb/lineups/22912.json")
print(f"\nLineup Schema:")
lineup_sample.printSchema()
print(f"\nSample records:")
lineup_sample.show(2, truncate=False)

In [0]:
# Explore matches folder structure
matches_path = "s3://matchpulse-pawan/raw/statsbomb/matches/"

print("\nMATCHES FOLDER STRUCTURE:")
print("="*70)

matches_items = dbutils.fs.ls(matches_path)
for item in matches_items:
    if item.name.endswith('/'):
        print(f"\n📂 Folder: {item.name}")
        # List files in this subfolder
        subfolder_files = dbutils.fs.ls(item.path)
        print(f"   Files: {len(subfolder_files)}")
        for file in subfolder_files[:10]:  # Show first 10
            size_kb = file.size / 1024
            print(f"     • {file.name} ({size_kb:.1f} KB)")
        if len(subfolder_files) > 10:
            print(f"     ... and {len(subfolder_files) - 10} more files")
    else:
        size_kb = item.size / 1024
        print(f"\n📄 File: {item.name} ({size_kb:.1f} KB)")

In [0]:
# Explore each statsbomb subfolder in detail
import os

def list_folder_recursive(path, prefix="", max_files=20):
    """List files in a folder with size and count"""
    try:
        items = dbutils.fs.ls(path)
        files = [item for item in items if not item.name.endswith('/')]
        folders = [item for item in items if item.name.endswith('/')]
        
        print(f"{prefix}📁 {path}")
        print(f"{prefix}  Files: {len(files)}, Folders: {len(folders)}")
        
        if files:
            print(f"{prefix}  Sample files (first {min(max_files, len(files))}):") 
            for i, file in enumerate(files[:max_files]):
                size_mb = file.size / (1024 * 1024)
                print(f"{prefix}    • {file.name} ({size_mb:.2f} MB)")
            if len(files) > max_files:
                print(f"{prefix}    ... and {len(files) - max_files} more files")
        
        return len(files), len(folders)
    except Exception as e:
        print(f"{prefix}  ❌ Error: {str(e)}")
        return 0, 0

# Explore statsbomb folders
base_path = "s3://matchpulse-pawan/raw/statsbomb/"
subfolders = ['events/', 'lineups/', 'matches/']

print("\n" + "="*70)
print("STATSBOMB DATA STRUCTURE")
print("="*70 + "\n")

for subfolder in subfolders:
    folder_path = base_path + subfolder
    file_count, folder_count = list_folder_recursive(folder_path)
    print()